In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from OSDE.LegendreExpSPDensity import LegExp, LegExpSPDensity, LegExpResult
from StocProcess.RBM import RBMTransProb, MakeRBMTransProbFunc
from QAE.RQAE import RQAE

In [2]:
# RBM parameters
c = -1
d = 1
x0 = 0.5 * (c + d)
t0 = 0
mu = 0.5
sigma = 1.0
n_terms = 5

tN = 0.6
t1 = 0.2

# approximation setting
maxDeg = 10
R = 12
eps0 = 1 / 2**8
integEpsabs = 1e-4
nRep = 10

In [3]:
Ns = np.tile((2 ** np.linspace(3, 6.5, 8)).astype(int), nRep)
print(Ns)

[ 8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90
  8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90
  8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90  8 11 16 22 32 45 64 90
  8 11 16 22 32 45 64 90]


In [4]:
# PDF at time t
transProbFunc = MakeRBMTransProbFunc(tN, t0, c, d, mu, sigma, n_terms)

# Prob(X > (c + d)/2)
integFunc = lambda x: transProbFunc(x, x0)
pTrue, _ = sp.integrate.quad(integFunc, x0, 1.0)

In [5]:
epss = []
pEsts = []
totalQueryNums = []
maxDepths = []
NsComp = []

for N in Ns:
    legExpResultPrev = None
    ts = np.concatenate([[t0], np.linspace(t1, tN, N)])
    eps = eps0 / np.sqrt(N)
    epss.append(eps)
    totalQueryNum = 0
    maxDepth = 0
    print(datetime.datetime.now(), "N=", N)

    for i in range(len(ts)-1):
        print("i_t=", i, datetime.datetime.now())
        transProbFunc = MakeRBMTransProbFunc(ts[i+1], ts[i], c, d, mu, sigma, n_terms)

        if i == 0:
            densFunc = lambda x: transProbFunc(x, x0)
            legExpResult = LegExp(densFunc, maxDeg)
        else:
            legExpResult = LegExpSPDensity(legExpResultPrev.fApp, transProbFunc, maxDeg, epsabs=integEpsabs)

        coefs = np.zeros(maxDeg+1)
        coefs[0] = 0.5

        for l in range(1, maxDeg+1):
            a = 0.5 * (legExpResult.coefs[l] / (l + 0.5) + 1)
            rqaeResult = RQAE(a, eps, R)
            coefs[l] = (2 * rqaeResult.aEst -1) * (l + 0.5)
            totalQueryNum += rqaeResult.TotalQueryNum
            maxDepth = max(maxDepth, rqaeResult.MaxDepth)

        legExpResultPrev = LegExpResult(coefs)

    pEsts.append(sp.integrate.quad(legExpResultPrev.fApp, x0, 1.0)[0])
    totalQueryNums.append(totalQueryNum)
    maxDepths.append(maxDepth)
    NsComp.append(N)

    retDf = pd.DataFrame(dict(N=NsComp,
                              pTrue=np.repeat(pTrue, len(NsComp)),
                              eps=epss,
                              pEst=pEsts,
                              absErr=np.abs(np.array(pEsts) - pTrue),
                              totalQueryNum=totalQueryNums,
                              maxDepth=maxDepths))
    retDf.to_csv('DivideRBM_RQAE.csv', index=False)

2025-01-31 16:48:48.931761 N= 8
i_t= 0 2025-01-31 16:48:48.931761


c:\Users\koich\Desktop\Code\DivQCOSDE\QAE\MaximizeL.py:11: RuntimeWarning: divide by zero encountered in log
  neglogL = lambda theta: -np.dot(n1s, np.log(np.sin(thetaMuls * theta)**2)) - np.dot(n0s, np.log(np.cos(thetaMuls * theta)**2))


i_t= 1 2025-01-31 16:48:51.203410
i_t= 2 2025-01-31 16:49:24.358965
i_t= 3 2025-01-31 16:49:49.746766
i_t= 4 2025-01-31 16:50:22.717931
i_t= 5 2025-01-31 16:50:51.447847
i_t= 6 2025-01-31 16:51:20.560157
i_t= 7 2025-01-31 16:51:56.898908
2025-01-31 16:52:18.816441 N= 11
i_t= 0 2025-01-31 16:52:18.817445
i_t= 1 2025-01-31 16:52:21.870834
i_t= 2 2025-01-31 16:52:54.960986
i_t= 3 2025-01-31 16:53:37.597168
i_t= 4 2025-01-31 16:54:08.150942
i_t= 5 2025-01-31 16:54:49.281989
i_t= 6 2025-01-31 16:55:24.327192
i_t= 7 2025-01-31 16:56:03.160359
i_t= 8 2025-01-31 16:56:38.012595
i_t= 9 2025-01-31 16:57:01.544929
i_t= 10 2025-01-31 16:57:32.621534
2025-01-31 16:58:07.587979 N= 16
i_t= 0 2025-01-31 16:58:07.589127
i_t= 1 2025-01-31 16:58:10.307339
i_t= 2 2025-01-31 16:58:58.674293
i_t= 3 2025-01-31 16:59:51.483612
i_t= 4 2025-01-31 17:00:32.634299
i_t= 5 2025-01-31 17:01:19.450378
i_t= 6 2025-01-31 17:01:59.000619
i_t= 7 2025-01-31 17:02:33.471752
i_t= 8 2025-01-31 17:03:11.059457
i_t= 9 2025-01-

KeyboardInterrupt: 

In [8]:
retDf

,N,pTrue,eps,pEst,absErr,totalQueryNum,maxDepth
0,8,0.649605,0.005524,0.645924,0.003681,162677,181
1,11,0.649605,0.004711,0.653987,0.004383,236337,212
2,16,0.649605,0.003906,0.646349,0.003256,363764,255
3,22,0.649605,0.003331,0.644986,0.004618,864915,300
4,32,0.649605,0.002762,0.644474,0.005131,1318996,362
5,45,0.649605,0.002329,0.649118,0.000487,1946967,429
6,64,0.649605,0.001953,0.644036,0.005568,2935148,511
7,90,0.649605,0.001647,0.651631,0.002026,7140622,607
8,128,0.649605,0.001381,0.653556,0.003951,10600092,724
9,8,0.649605,0.005524,0.657206,0.007602,164164,181


In [9]:
retDf.groupby('N')['absErr'].mean()

N
8      0.003221
11     0.002877
16     0.002774
22     0.002475
32     0.002233
45     0.001315
64     0.002383
90     0.001689
128    0.003843
Name: absErr, dtype: float64